In [3]:
import random

def roll_d20():
    """Roll a 20-sided die."""
    return random.randint(1, 20)

def roll_dice(num_dice, sides):
    """Roll multiple dice and return the total. E.g., roll_dice(2, 6) for 2d6."""
    return sum(random.randint(1, sides) for _ in range(num_dice))

In [4]:
class Character:
    """Base class for all characters in the game."""
    
    def __init__(self, name, health, strength, defense):
        self.name = name
        self.max_health = health
        self.health = health
        self.strength = strength
        self.defense = defense
        self.infection_level = 0  # 0-100 scale, where 100 is fully infected
    
    def is_alive(self):
        """Return True if the character is still alive."""
        return self.health > 0
    
    def take_damage(self, amount):
        """Reduce health by damage amount, but don't go below 0."""
        self.health = max(0, self.health - amount)
        print(f"{self.name} takes {amount} damage! (HP: {self.health}/{self.max_health})")
        
        if self.infection_level > 0:
            status = "🦠" if self.infection_level >= 50 else "⚠️"
            print(f"{status} Infection level: {self.infection_level}%")
        
        if not self.is_alive():
            print(f"{self.name} has fallen!")
    
    def apply_infection(self, amount):
        """Apply infection damage. At 100%, character is incapacitated."""
        self.infection_level = min(100, self.infection_level + amount)
        if amount > 0:
            print(f"🦠 {self.name} becomes infected! (Infection: {self.infection_level}%)")
        if self.infection_level >= 100:
            print(f"💀 {self.name} has succumbed to infection!")
            self.health = 0
    
    def attack(self, target):
        """Perform a D20 attack against a target."""
        print(f"\n{self.name} attacks {target.name}!")
        
        attack_roll = roll_d20()
        total_attack = attack_roll + self.strength
        
        print(f"Roll: {attack_roll} + STR({self.strength}) = {total_attack}")
        print(f"{target.name}'s Defense: {target.defense}")
        
        if total_attack > target.defense:
            print("Hit!")
            damage = self.strength
            target.take_damage(damage)
        else:
            print("Miss!")
    
    def __str__(self):
        return f"{self.name} (HP: {self.health}/{self.max_health}, STR: {self.strength}, DEF: {self.defense})"

In [5]:
class Weapon:
    """Represents a weapon that can boost damage."""
    
    def __init__(self, name, min_damage, max_damage, description=""):
        self.name = name
        self.min_damage = min_damage
        self.max_damage = max_damage
        self.description = description
    
    def get_damage(self):
        """Roll random damage between min and max."""
        return random.randint(self.min_damage, self.max_damage)
    
    def __str__(self):
        return f"{self.name} (Damage: {self.min_damage}-{self.max_damage})"


# Define all weapon types available in the game
WEAPONS = {
    # Early game melee weapons
    "Rusty Bat": Weapon("Rusty Bat", 2, 4, "An old baseball bat covered in rust"),
    "Tire Iron": Weapon("Tire Iron", 2, 5, "A heavy metal tire iron"),
    "Screwdriver": Weapon("Screwdriver", 1, 3, "A sharp flathead screwdriver"),
    "Pipe": Weapon("Pipe", 3, 5, "A sturdy metal pipe"),
    "Crowbar": Weapon("Crowbar", 3, 6, "A classic survivor's tool"),
    
    # Mid-tier firearms
    "Handgun": Weapon("Handgun", 5, 8, "A 9mm handgun with limited ammo"),
    "Mac-10": Weapon("Mac-10", 6, 9, "A compact submachine gun"),
    "Revolver": Weapon("Revolver", 5, 9, "A .357 magnum revolver"),
    
    # High-tier weapons
    "Shotgun": Weapon("Shotgun", 8, 12, "A pump-action shotgun"),
    "AK-47": Weapon("AK-47", 7, 10, "A reliable assault rifle"),
    "AR-15": Weapon("AR-15", 7, 11, "A modern assault rifle"),
    "Desert Eagle": Weapon("Desert Eagle", 6, 10, "A powerful .50 caliber handgun"),
}

In [6]:
class Player(Character):
    """The player character."""
    
    def __init__(self, name):
        super().__init__(name, health=100, strength=8, defense=12)
        self.inventory = []
        self.xp = 0
        self.crit_chance = 0.20
        self.equipped_weapon = None
    
    def pick_up(self, item):
        """Add an item to inventory."""
        self.inventory.append(item)
        weapon_indicator = " ⚔️ [WEAPON]" if item in WEAPONS else ""
        print(f"\n✅ You picked up: {item}{weapon_indicator}")
        if item in WEAPONS:
            print(f"💡 Tip: Type 'equip {item.lower()}' to use this weapon in combat")
    
    def show_inventory(self):
        """Display inventory contents."""
        print(f"\n{'='*50}")
        print(f"📦 {self.name}'s Inventory")
        print(f"{'='*50}")
        
        if self.equipped_weapon:
            print(f"⚔️  Equipped: {self.equipped_weapon}")
        else:
            print(f"⚔️  Equipped: Fists (Damage: {self.strength})")
        
        print()
        if not self.inventory:
            print("Inventory is empty.")
        else:
            print("Items:")
            for item in self.inventory:
                weapon_marker = " [WEAPON]" if item in WEAPONS else ""
                print(f"  - {item}{weapon_marker}")
        print(f"{'='*50}")
    
    def equip_weapon(self, weapon_name):
        """Equip a weapon from inventory."""
        if weapon_name not in self.inventory:
            print(f"You don't have '{weapon_name}' in your inventory.")
            return
        
        if weapon_name not in WEAPONS:
            print(f"'{weapon_name}' is not a weapon.")
            return
        
        self.equipped_weapon = WEAPONS[weapon_name]
        print(f"✓ {self.name} equipped {self.equipped_weapon}")
    
    def attack(self, target):
        """Perform a D20 attack with critical hit chance and weapon damage."""
        print(f"\n{self.name} attacks {target.name}!")
        
        attack_roll = roll_d20()
        total_attack = attack_roll + self.strength
        
        print(f"Roll: {attack_roll} + STR({self.strength}) = {total_attack}")
        print(f"{target.name}'s Defense: {target.defense}")
        
        is_critical = random.random() < self.crit_chance
        
        if total_attack > target.defense:
            base_damage = self.strength
            weapon_damage = 0
            if self.equipped_weapon:
                weapon_damage = self.equipped_weapon.get_damage()
                print(f"💥 Weapon bonus: +{weapon_damage} from {self.equipped_weapon.name}!")
            
            total_damage = base_damage + weapon_damage
            
            if is_critical:
                total_damage = total_damage * 2
                print(f"⚡ CRITICAL HIT! {self.name} strikes with devastating force!")
            else:
                print("Hit!")
            
            target.take_damage(total_damage)
        else:
            print("Miss!")


class Enemy(Character):
    """Base class for enemies."""
    
    def __init__(self, name, health, strength, defense, xp_value=10, loot=None, infection_chance=0.15):
        super().__init__(name, health, strength, defense)
        self.xp_value = xp_value
        self.loot = loot if loot else []
        self.infection_chance = infection_chance
    
    def attack(self, target):
        """Enemies attack with flavor text and infection chance."""
        print(f"\n{self.name} lunges at {target.name}!")
        
        attack_roll = roll_d20()
        total_attack = attack_roll + self.strength
        
        if total_attack > target.defense:
            damage = roll_dice(1, 4) + self.strength
            print(f"{self.name} claws viciously!")
            target.take_damage(damage)
            
            if random.random() < self.infection_chance:
                infection_amount = roll_dice(1, 3) + 5
                target.apply_infection(infection_amount)
        else:
            print(f"{self.name}'s attack misses!")


class Minion(Enemy):
    """Weak zombie."""
    def __init__(self):
        super().__init__("Walker", health=30, strength=4, defense=8, xp_value=15, loot=["Rotten Food"], infection_chance=0.20)


class Elite(Enemy):
    """Tougher, faster enemy."""
    def __init__(self):
        super().__init__("Runner", health=50, strength=7, defense=11, xp_value=30, loot=["Medkit", "Ammo"], infection_chance=0.25)


class Boss(Enemy):
    """Final boss enemy."""
    def __init__(self):
        super().__init__("Mutated Alpha", health=120, strength=12, defense=14, xp_value=100, loot=["Radio Part", "Rare Weapon"], infection_chance=0.35)

In [7]:
class Location:
    """A location in the game world."""
    
    def __init__(self, name, description):
        self.name = name
        self.description = description
        self.connections = {}
        self.enemies = []
        self.items = []
    
    def describe(self):
        """Print a full description of the location."""
        print(f"\n{'='*50}")
        print(f"📍 {self.name}")
        print(f"{'='*50}")
        print(self.description)
        
        if self.enemies:
            print("\n⚠️ ENEMIES HERE:")
            for enemy in self.enemies:
                print(f"  - {enemy.name} (HP: {enemy.health}/{enemy.max_health})")
        
        if self.items:
            print("\n💰 ITEMS YOU CAN PICK UP:")
            for item in self.items:
                weapon_indicator = " [WEAPON]" if item in WEAPONS else ""
                print(f"  - {item}{weapon_indicator}")
        
        print(f"\n🧭 DIRECTIONS YOU CAN TRAVEL:")
        if self.connections:
            for direction, destination in self.connections.items():
                print(f"  → {direction.upper()} - leads to {destination.name}")
        else:
            print("  (No exits available)")
        
        print("\n" + "="*50)
        print("❓ WHAT DO YOU DO?")
        print("="*50)
        
        if self.items:
            for item in self.items[:2]:
                print(f"   → take {item.lower()}")
        
        if self.connections:
            directions_list = ", ".join(list(self.connections.keys())[:3])
            print(f"🗺️  Explore: {directions_list}")
        
        if self.enemies:
            print(f"⚔️  Combat: fight")
        
        print(f"📋 Other: inventory, look, help, quit")
        print("="*50)
    
    def get_exits(self):
        """Return a list of available directions."""
        return list(self.connections.keys())
    
    def add_connection(self, direction, location):
        """Connect this location to another."""
        self.connections[direction] = location

In [8]:
def create_world():
    """Create and connect all locations. Returns the starting location."""
    
    safehouse = Location("Suburban Safehouse", "A barricaded residential home serving as your base of operations.\nBoarded windows and reinforced doors keep the undead at bay.")
    downtown = Location("Downtown Core", "Once a bustling shopping district, now a ghost town.\nOverturned cars and broken storefronts line the streets.")
    radio_tower = Location("Radio Tower - Extraction Point", "A tall communications tower looms overhead.\nIf you can repair the radio, the evacuation helicopter will arrive.")
    subway = Location("Subway Tunnels", "Dark, damp tunnels stretch beneath the city.\nEmergency lights flicker ominously.")
    park = Location("Overgrown Park", "Nature has begun reclaiming this abandoned park.\nVines creep up the playground equipment.")
    
    # Connect locations
    downtown.add_connection("south", safehouse)
    safehouse.add_connection("north", downtown)
    downtown.add_connection("north", radio_tower)
    radio_tower.add_connection("south", downtown)
    downtown.add_connection("east", subway)
    subway.add_connection("west", downtown)
    downtown.add_connection("west", park)
    park.add_connection("east", downtown)
    
    # Add enemies
    downtown.enemies.append(Elite())
    downtown.enemies.append(Minion())
    subway.enemies.append(Elite())
    subway.enemies.append(Elite())
    park.enemies.append(Minion())
    radio_tower.enemies.append(Boss())
    
    # Add items
    safehouse.items.extend(["Rusty Bat", "Tire Iron", "Bandage"])
    park.items.extend(["Screwdriver", "Medkit", "Canned Food"])
    downtown.items.extend(["Pipe", "Crowbar", "Handgun", "Ammo", "Keycard"])
    subway.items.extend(["Mac-10", "Shotgun", "Revolver", "Flashlight", "Radio Part"])
    radio_tower.items.extend(["AK-47", "AR-15", "Desert Eagle"])
    
    return safehouse

In [9]:
class Combat:
    """Manages turn-based combat between player and enemy."""
    
    PLAYER_TURN = "player_turn"
    ENEMY_TURN = "enemy_turn"
    COMBAT_END = "combat_end"
    
    def __init__(self, player, enemy):
        self.player = player
        self.enemy = enemy
        self.state = Combat.PLAYER_TURN
        self.player_guarding = False
    
    def start(self):
        """Begin combat and run until someone wins/loses/flees."""
        print(f"\n⚔️ COMBAT BEGINS! ⚔️")
        print(f"{self.player.name} vs {self.enemy.name}!")
        print("="*60)
        
        while self.state != Combat.COMBAT_END:
            if self.state == Combat.PLAYER_TURN:
                self.player_turn()
            elif self.state == Combat.ENEMY_TURN:
                self.enemy_turn()
        
        return self.get_result()
    
    def player_turn(self):
        """Handle player's turn in combat."""
        print(f"\n{self.player} | {self.enemy}")
        print("\nWhat do you do?")
        print("  (a)ttack")
        print("  (d)efend")
        print("  (r)un")
        
        action = input("> ").lower().strip()
        
        if action in ["a", "attack"]:
            self.player.attack(self.enemy)
            if self.enemy.is_alive():
                self.state = Combat.ENEMY_TURN
            else:
                print(f"\n🎉 {self.enemy.name} has been defeated!")
                self.state = Combat.COMBAT_END
        
        elif action in ["d", "defend"]:
            self.player_guarding = True
            print(f"\n🛡️  {self.player.name} takes a defensive stance!")
            print("Incoming damage will be reduced this turn.")
            self.state = Combat.ENEMY_TURN
        
        elif action in ["r", "run"]:
            self.attempt_escape()
        
        else:
            print("Invalid action. Try (a)ttack, (d)efend, or (r)un.")
    
    def attempt_escape(self):
        """Attempt to flee from combat."""
        print(f"\n{self.player.name} attempts to flee!")
        
        escape_chance = 0.5
        if self.player.infection_level > 50:
            escape_chance -= 0.15
            print("🦠 Your infection slows you down...")
        
        escape_chance = max(0.1, escape_chance)
        
        if random.random() < escape_chance:
            print("✅ You successfully fled from combat!")
            self.state = Combat.COMBAT_END
        else:
            print("❌ You couldn't get away!")
            self.state = Combat.ENEMY_TURN
    
    def enemy_turn(self):
        """Handle enemy's turn in combat."""
        print(f"\n{self.enemy.name}'s turn...\n")
        
        # Apply damage reduction if player is guarding
        if self.player_guarding:
            print(f"🛡️  {self.player.name} is defending...")
        
        self.resolve_enemy_attack()
        self.player_guarding = False  # Guard only lasts one turn
        
        if not self.player.is_alive():
            print(f"\n💀 {self.player.name} has fallen!")
            self.state = Combat.COMBAT_END
        else:
            self.state = Combat.PLAYER_TURN
    
    def resolve_enemy_attack(self):
        """Resolve enemy attack with damage reduction for guarding."""
        print(f"{self.enemy.name} attacks {self.player.name}!")
        
        attack_roll = roll_d20()
        total_attack = attack_roll + self.enemy.strength
        
        if total_attack > self.player.defense:
            base_damage = roll_dice(1, 4) + self.enemy.strength
            
            if self.player_guarding:
                damage = max(1, base_damage // 2)  # Half damage when guarding
                print(f"{self.enemy.name} claws viciously!")
                print(f"🛡️  You block most of the damage! ({base_damage} → {damage})")
            else:
                damage = base_damage
                print(f"{self.enemy.name} claws viciously!")
            
            self.player.take_damage(damage)
            
            if random.random() < self.enemy.infection_chance:
                infection_amount = roll_dice(1, 3) + 5
                self.player.apply_infection(infection_amount)
        else:
            print(f"{self.enemy.name}'s attack misses!")
    
    def get_result(self):
        """Return the combat result."""
        if not self.enemy.is_alive():
            return "victory"
        elif not self.player.is_alive():
            return "defeat"
        else:
            return "fled"

In [10]:
class Game:
    """Main game controller - handles exploration and combat modes."""
    
    # Game states
    EXPLORING = "exploring"
    IN_COMBAT = "in_combat"
    GAME_OVER = "game_over"
    VICTORY = "victory"
    
    def __init__(self):
        self.player = None
        self.current_location = None
        self.state = Game.EXPLORING
        self.game_running = True
    
    def start(self):
        """Initialize and start the game."""
        self.show_intro()
        self.create_player()
        self.current_location = create_world()
        self.current_location.describe()
        
        # Main game loop
        while self.game_running:
            if self.state == Game.EXPLORING:
                self.exploration_loop()
            elif self.state == Game.GAME_OVER:
                self.show_game_over()
                break
            elif self.state == Game.VICTORY:
                self.show_victory()
                break
    
    def show_intro(self):
        """Display the game introduction."""
        print("\n" + "="*60)
        print("    🧟 ZOMBIE SURVIVAL: TEXT ADVENTURE 🧟")
        print("="*60)
        print("\nYou wake up in a barricaded safehouse.")
        print("The city outside is overrun with undead.")
        print("Your goal: Reach the radio tower, repair it, and escape!")
        print("="*60)
    
    def create_player(self):
        """Create the player character."""
        print("\nWhat is your name, survivor?")
        name = input("> ").strip()
        if not name:
            name = "Survivor"
        self.player = Player(name)
        print(f"\nWelcome, {name}! Your adventure begins...")
        print(f"HP: {self.player.health} | STR: {self.player.strength} | DEF: {self.player.defense}")
    
    def exploration_loop(self):
        """Handle player input during exploration (non-combat) mode."""
        self.show_actions()
        command = input("> ").lower().strip()
        
        if not command:
            return
        
        parts = command.split()
        action = parts[0]
        
        if action == "help":
            self.show_help()
        
        elif action == "look":
            self.current_location.describe()
        
        elif action == "go" and len(parts) > 1:
            direction = parts[1]
            self.move(direction)
        
        elif action in ["north", "south", "east", "west", "up", "down"]:
            self.move(action)
        
        elif action in ["fight", "attack"]:
            self.initiate_combat()
        
        elif action in ["inventory", "i"]:
            self.player.show_inventory()
        
        elif action in ["take", "grab"]:
            item_name = " ".join(parts[1:]) if len(parts) > 1 else None
            self.pick_up_item(item_name)
        
        elif action == "pick" and len(parts) > 1 and parts[1] == "up":
            item_name = " ".join(parts[2:]) if len(parts) > 2 else None
            self.pick_up_item(item_name)
        
        elif action == "pickup":
            item_name = " ".join(parts[1:]) if len(parts) > 1 else None
            self.pick_up_item(item_name)
        
        elif action == "get":
            item_name = " ".join(parts[1:]) if len(parts) > 1 else None
            self.pick_up_item(item_name)
        
        elif action in ["equip", "wield"]:
            weapon_name = " ".join(parts[1:]) if len(parts) > 1 else None
            self.equip_weapon(weapon_name)
        
        elif action == "quit":
            print("Thanks for playing!")
            self.game_running = False
        
        else:
            print("I don't understand that command. Type 'help' for options.")
    
    def show_actions(self):
        """Show available commands with context-specific suggestions."""
        print("\n" + "="*60)
        print("⚡ WHAT DO YOU DO?")
        print("="*60)
        
        if self.current_location.items:
            print(f"\n📦 PICK UP ITEMS:")
            for item in self.current_location.items[:2]:
                print(f"   take {item.lower()}")
        
        available_dirs = list(self.current_location.connections.keys())
        if available_dirs:
            print(f"\n🗺️  EXPLORE:")
            for direction in available_dirs[:3]:
                dest = self.current_location.connections[direction]
                print(f"   {direction}  (go to {dest.name})")
        
        if self.current_location.enemies:
            print(f"\n⚔️  COMBAT:")
            enemy_names = [enemy.name for enemy in self.current_location.enemies]
            print(f"   fight  (encounter {', '.join(enemy_names)})")
        
        print(f"\n🎒 INVENTORY:")
        print(f"   inventory  (i)  - Check your items")
        if self.player.inventory:
            weapons = [item for item in self.player.inventory if item in WEAPONS]
            if weapons:
                print(f"   equip [weapon]")
        
        print(f"\n❓ OTHER:")
        print(f"   look")
        print(f"   help")
        print(f"   quit")
        print("="*60)
    
    def move(self, direction):
        """Move the player in the specified direction."""
        if direction in self.current_location.connections:
            self.current_location = self.current_location.connections[direction]
            print(f"\n🚶 You travel {direction}...")
            self.current_location.describe()
        else:
            print(f"❌ You can't go {direction} from here.")
    
    def pick_up_item(self, item_name=None):
        """Pick up an item from the current location."""
        if not self.current_location.items:
            print("❌ There are no items to pick up here.")
            return
        
        if not item_name:
            print("\n📦 Items available here:")
            for item in self.current_location.items:
                weapon_marker = " [WEAPON]" if item in WEAPONS else ""
                print(f"  - {item}{weapon_marker}")
            return
        
        # Case-insensitive matching
        matched_item = None
        for item in self.current_location.items:
            if item.lower() == item_name.lower():
                matched_item = item
                break
        
        if not matched_item:
            print(f"❌ There is no '{item_name}' here.")
            return
        
        self.current_location.items.remove(matched_item)
        self.player.pick_up(matched_item)
    
    def equip_weapon(self, weapon_name=None):
        """Equip a weapon from the player's inventory."""
        if not weapon_name:
            print("\n🎒 Available weapons in inventory:")
            weapons_found = False
            for item in self.player.inventory:
                if item in WEAPONS:
                    print(f"  - {item} ({WEAPONS[item]})")
                    weapons_found = True
            
            if not weapons_found:
                print("  (No weapons in inventory)")
            return
        
        # Case-insensitive matching
        matched_weapon = None
        for item in self.player.inventory:
            if item.lower() == weapon_name.lower():
                matched_weapon = item
                break
        
        if not matched_weapon:
            print(f"❌ You don't have '{weapon_name}' in your inventory.")
            return
        
        self.player.equip_weapon(matched_weapon)
    
    def initiate_combat(self):
        """Start combat with an enemy in the current location."""
        if not self.current_location.enemies:
            print("❌ There's nothing to fight here.")
            return
        
        enemy = self.current_location.enemies[0]
        
        print(f"\n⚠️ A wild {enemy.name} appears!")
        print(f"Enemy HP: {enemy.health}/{enemy.max_health}")
        print(f"Enemy DEF: {enemy.defense}")
        
        if self.player.equipped_weapon:
            print(f"Your weapon: {self.player.equipped_weapon}")
        else:
            print(f"You're fighting with your fists (STR: {self.player.strength})")
        
        # Enter turn-based combat only during combat
        self.state = Game.IN_COMBAT
        battle = Combat(self.player, enemy)
        result = battle.start()
        
        # Return to exploring after combat ends
        self.state = Game.EXPLORING
        
        if result == "victory":
            self.current_location.enemies.remove(enemy)
            print(f"\n✅ You defeated the {enemy.name}!")
            if enemy.loot:
                print(f"You found some loot!")
            self.check_victory_condition(enemy)
        
        elif result == "defeat":
            self.state = Game.GAME_OVER
        
        elif result == "fled":
            print(f"\n🏃 You managed to escape back to the {self.current_location.name}")
    
    def check_victory_condition(self, enemy):
        """Check if the player meets the win conditions."""
        if not isinstance(enemy, Boss):
            return
        
        at_radio_tower = "Radio Tower" in self.current_location.name
        has_radio_part = "Radio Part" in self.player.inventory
        has_keycard = "Keycard" in self.player.inventory
        
        if at_radio_tower and has_radio_part and has_keycard:
            print("\n📡 You repair the radio and call for extraction!")
            print("🚁 The evac chopper arrives just as the horde closes in...")
            self.state = Game.VICTORY
        else:
            print("\n⚠️ You defeated the boss, but extraction isn't ready yet.")
            if not has_radio_part:
                print("  Missing: Radio Part")
            if not has_keycard:
                print("  Missing: Keycard")
            if not at_radio_tower:
                print("  You must defeat the boss at the Radio Tower")
    
    def show_help(self):
        """Display available commands."""
        print("\n" + "="*70)
        print("📜 AVAILABLE COMMANDS:")
        print("="*70)
        print("\n  MOVEMENT:")
        print("    go [direction]     - Move in a direction")
        print("    north/south/east/west - Direct movement")
        print("\n  ITEMS:")
        print("    take [item]        - Pick up an item")
        print("    get [item]         - Pick up an item")
        print("    pick up [item]     - Pick up an item")
        print("    grab [item]        - Pick up an item")
        print("\n  INVENTORY & EQUIPMENT:")
        print("    inventory (i)      - Check your inventory")
        print("    equip [weapon]     - Equip a weapon")
        print("\n  COMBAT:")
        print("    fight              - Attack an enemy")
        print("    (During combat: attack, defend, run)")
        print("\n  OTHER:")
        print("    look               - Examine your surroundings")
        print("    help               - Show this help message")
        print("    quit               - Exit the game")
        print("="*70)
    
    def show_game_over(self):
        """Display game over message."""
        print("\n" + "="*60)
        print("💀 GAME OVER - YOU HAVE FALLEN 💀")
        print("="*60)
        print("\nYou have succumbed to the undead infection...")
        print("The zombie apocalypse claims another victim.")
        print("\nFinal Stats:")
        print(f"  Location: {self.current_location.name}")
        print(f"  Health: {self.player.health}/{self.player.max_health}")
        print(f"  Infection: {self.player.infection_level}%")
        print(f"  Items: {len(self.player.inventory)}")
        print("="*60)
    
    def show_victory(self):
        """Display victory message."""
        print("\n" + "="*60)
        print("🎉 VICTORY! YOU HAVE ESCAPED! 🎉")
        print("="*60)
        print(f"\nCongratulations, {self.player.name}!")
        print("You successfully repaired the radio and escaped!")
        print("The helicopter lifted you to safety just in time.")
        print("\nFinal Stats:")
        print(f"  Health: {self.player.health}/{self.player.max_health}")
        print(f"  Infection: {self.player.infection_level}%")
        print(f"  Items Collected: {len(self.player.inventory)}")
        print("="*60)

In [11]:
game = Game()
game.start()


    🧟 ZOMBIE SURVIVAL: TEXT ADVENTURE 🧟

You wake up in a barricaded safehouse.
The city outside is overrun with undead.
Your goal: Reach the radio tower, repair it, and escape!

What is your name, survivor?

Welcome, Carlo! Your adventure begins...
HP: 100 | STR: 8 | DEF: 12

📍 Suburban Safehouse
A barricaded residential home serving as your base of operations.
Boarded windows and reinforced doors keep the undead at bay.

💰 ITEMS YOU CAN PICK UP:
  - Rusty Bat [WEAPON]
  - Tire Iron [WEAPON]
  - Bandage

🧭 DIRECTIONS YOU CAN TRAVEL:
  → NORTH - leads to Downtown Core

❓ WHAT DO YOU DO?
   → take rusty bat
   → take tire iron
🗺️  Explore: north
📋 Other: inventory, look, help, quit

⚡ WHAT DO YOU DO?

📦 PICK UP ITEMS:
   take rusty bat
   take tire iron

🗺️  EXPLORE:
   north  (go to Downtown Core)

🎒 INVENTORY:
   inventory  (i)  - Check your items

❓ OTHER:
   look
   help
   quit

📍 Suburban Safehouse
A barricaded residential home serving as your base of operations.
Boarded windows 